In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
import xgboost as xgb


In [ ]:
data = pd.read_csv(r"/content/Train-Set.csv")
data

In [ ]:
data.info()

In [ ]:
22870/len(data)

In [ ]:
data["contact"].value_counts()

In [ ]:
data.head(10)  # ["job", "marital", "education", "contact", "poutcome"]

In [ ]:
data.isna().sum()

In [ ]:
yes_count = 0
no_count = 0
for i in range(len(data)):
  if not(pd.isna(data.loc[i, "balance"])) and (data.loc[i, "housing"] == "yes"):
    yes_count += data.loc[i, "balance"]
  elif not(pd.isna(data.loc[i, "balance"])) and (data.loc[i, "housing"] == "no"):
    no_count += data.loc[i, "balance"]

for j in range(len(data)):
  if data.loc[j, "housing"] == "yes":
    data.loc[j, "balance"] = yes_count / len(data[data["housing"] == "yes"])
  elif data.loc[j, "housing"] == "no":
    data.loc[j, "balance"] = no_count / len(data[data["housing"] == "no"])
  elif data.loc[j, "housing"] == "unknown":
    data.loc[j, "balance"] = 0.0

In [ ]:
data.isna().sum()

In [ ]:
data.corr(numeric_only=True)

In [ ]:
cols = ["default", "loan", "housing", "Target"]

for col in cols:
  data[col] = LabelEncoder().fit_transform(data[col])

In [ ]:
cat_cols = ["job", "marital", "education", "contact", "poutcome"]

encoded_cols = pd.get_dummies(data[cat_cols])

data = pd.concat([data.drop(cat_cols, axis=1), encoded_cols], axis=1)

In [ ]:
data.drop(["id", "day", "month"], axis=1, inplace=True)
data.info()

In [ ]:
'''data['loan'] = pd.to_numeric(data['loan'], errors='coerce')
data['Target'] = pd.to_numeric(data['Target'], errors='coerce')
data['housing'] = pd.to_numeric(data['housing'], errors='coerce')
data['default'] = pd.to_numeric(data['default'], errors='coerce')'''

In [ ]:
data.corr(numeric_only=True)

In [ ]:
corr = data.corr(numeric_only=True)

f, ax = plt.subplots(figsize=(30, 25))
sns.heatmap(corr, vmax=0.9, square=True, cmap='viridis', linewidths=0.5, xticklabels=corr.columns, yticklabels=corr.columns, annot=True)
plt.show()


In [ ]:
count=0
for col in data.columns:
  if  -0.1 < data[col].corr(data["Target"]) > 0.1:
    count+=1
print(count)

In [ ]:
X = data.drop("Target", axis=1)
y = data["Target"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
gb_clf = GradientBoostingClassifier(learning_rate= 0.1, max_depth= 5, n_estimators= 50, random_state=42)

gb_clf.fit(X_train, y_train)

In [ ]:
y_train_pred = gb_clf.predict(X_train)

y_pred = gb_clf.predict(X_val)

f1_score(y_train, y_train_pred, average="weighted")
f1_score(y_val, y_pred, average="weighted")

In [ ]:
model = xgb.XGBClassifier()
model.fit(X_train, y_train)
predictions = model.predict(X_val)

print(f1_score(y_val, predictions, average="weighted"))